# 20 · Where does the score go wrong on A2: missing features or a model that does not learn?

**Exploratory.** The A2 final set is used to *diagnose* only: nothing here is fitted on it (D46), and any feature designed from these errors must be confirmed on a different set (rule 8). Output is aggregate; no resume text is printed.

**Check 1 · train-fit vs held-out gap** (D50 labels, A2 train region). If a learner's AUC on its own training JDs is no higher than on held-out JDs, the 11 features cannot separate the labels (features missing). If train is far above held-out, it is overfitting or the labels are noisy.

**Check 2 · error taxonomy on the A2 final set.** Every (Fit, No Fit) pair inside a JD is an *inversion* when the No Fit scores higher. Inversions are split by what the pair is: role match (the role model's probability, in role if >= 0.5), which grade the Fit side has, and how reliable the label is.

In [1]:
import matplotlib.pyplot as plt
import re

import numpy as np
import pandas as pd

from candidate_screener import reporting
from candidate_screener.evaluation import a1_weights as aw, final_set, indomain
from candidate_screener.matching import corpus

reporting.style(plt)
frame = final_set.load().set_index("pair_id"); lab = indomain.labels(frame.reset_index())
a2 = corpus.a2_pairs(text_variant="v2").set_index("pair_id").loc[lab.index].reset_index()
d50, y50 = aw.d50_pairs()                    # load the big train documents before the model exists (peak memory)
model = aw.build_scorer()
X_a2 = aw.features(model, a2).join(aw.extra_features(model, a2))
X50 = aw.features(model, d50).join(aw.extra_features(model, d50))
del model, a2, d50["jd_text"], d50["cv_text"]; import gc; gc.collect()     # features are all that is needed from here
print(X_a2.shape, X50.shape)

(600, 11) (1000, 11)


## Check 1 · train vs held-out (D50, 70 train JDs / 30 held-out, 20 repetitions)

In [2]:
gap = aw.fit_gap(X50, y50, d50.jd_id)
g = gap.groupby("scorer")[["train AUC", "held-out AUC"]].mean()
g["gap"] = g["train AUC"] - g["held-out AUC"]
g.round(3)

,train AUC,held-out AUC,gap
scorer,,,
boosting all (11),0.915,0.845,0.070
hand-set X2,0.868,0.861,0.007
logistic all (11),0.869,0.855,0.014
logistic base (4),0.874,0.861,0.013


## Check 2 · inversions on the A2 final set

In [3]:
final = {"hand-set X2": aw.linear_score(X_a2, aw.HAND_WEIGHTS)}
for name, (kind, cols) in aw.LEARNERS.items():
    if name != "logistic base (4)":
        final[name + " | D50"] = aw.fit_learner(kind, cols, X50, y50)(X_a2)
inv = {n: aw.pair_inversions(s, lab.y, lab.jd_id) for n, s in final.items()}
rm = X_a2.role_match
grade = frame[["yc_raw", "van_raw"]].apply(lambda r: "Good" if "Good Fit" in r.values else ("Potential" if "Potential Fit" in r.values else "other"), axis=1)
def annotate(d):
    return d.assign(same_role=(rm.loc[d.fit_id].to_numpy() == rm.loc[d.nofit_id].to_numpy()),
                    fit_role=rm.loc[d.fit_id].to_numpy(), nofit_role=rm.loc[d.nofit_id].to_numpy(),
                    grade=grade.loc[d.fit_id].to_numpy(), tier=frame.tier.loc[d.fit_id].to_numpy(),
                    stratum=lab.stratum.loc[d.fit_id].to_numpy())
inv = {n: annotate(d) for n, d in inv.items()}
base = inv["hand-set X2"]
print(f"pairs: {len(base):,} (Fit, No Fit) pairs in {base.jd_id.nunique()} JDs; overall inversion rate {base.inv.mean():.3f}")
pd.DataFrame({n: {"inversion rate": d.inv.mean()} for n, d in inv.items()}).T.round(3)

pairs: 1,143 (Fit, No Fit) pairs in 59 JDs; overall inversion rate 0.148


,inversion rate
hand-set X2,0.148
logistic all (11) | D50,0.158
boosting all (11) | D50,0.150


In [4]:
def split(col):
    t = pd.DataFrame({n: d.groupby(col).inv.mean() for n, d in inv.items()})
    t.insert(0, "n pairs", base.groupby(col).size()); return t.round(3)
# 1. does the error sit between role families or inside one?
# role_match is the role model's probability, so "in role" means >= 0.5
fi, ni = base.fit_role >= 0.5, base.nofit_role >= 0.5
by_role = base.assign(kind=np.select([fi & ~ni, ~fi & ni, fi & ni], ["Fit in role, No Fit out", "Fit out of role, No Fit in", "both in role"], "both out of role"))
t = pd.DataFrame({n: d.assign(kind=by_role.kind).groupby("kind").inv.mean() for n, d in inv.items()})
t.insert(0, "n pairs", by_role.groupby("kind").size()); t.insert(1, "share of all inversions", (by_role.groupby("kind").inv.sum() / base.inv.sum()).round(3))
t.round(3)

,n pairs,share of all inversions,hand-set X2,logistic all (11) | D50,boosting all (11) | D50
kind,,,,,
"Fit in role, No Fit out",452,0.006,0.002,0.000,0.000
"Fit out of role, No Fit in",16,0.089,0.938,0.938,0.938
both in role,78,0.195,0.423,0.397,0.449
both out of role,597,0.710,0.201,0.226,0.204


In [5]:
print("by grade of the Fit side (Good = a human said Good Fit)"); display(split("grade"))
print("by label reliability"); display(split("tier"))
print("by stratum"); split("stratum")

by grade of the Fit side (Good = a human said Good Fit)


,n pairs,hand-set X2,logistic all (11) | D50,boosting all (11) | D50
grade,,,,
Good,617,0.130,0.135,0.123
Potential,526,0.169,0.186,0.182


by label reliability


,n pairs,hand-set X2,logistic all (11) | D50,boosting all (11) | D50
tier,,,,
humans_agree,416,0.135,0.149,0.142
llm_breaks_tie,239,0.272,0.285,0.282
single_human,488,0.098,0.105,0.092


by stratum


,n pairs,hand-set X2,logistic all (11) | D50,boosting all (11) | D50
stratum,,,,
generic,722,0.129,0.133,0.116
targeted,421,0.181,0.202,0.208


### What the misordered documents look like (feature means; no text)

In [6]:
cols = ["role_match", "experience_compatible", "tfidf_scaled", "coverage", "coverage_unweighted", "log_jd_concepts", "log_cv_concepts", "exp_known"]
miss = base.groupby("fit_id").inv.mean(); false_alarm = base.groupby("nofit_id").inv.mean()
rows = {"Fit, ranked correctly (miss < .5)": X_a2.loc[miss[miss < .5].index, cols].mean(),
        "Fit, missed (miss > .5)": X_a2.loc[miss[miss > .5].index, cols].mean(),
        "No Fit, ranked correctly": X_a2.loc[false_alarm[false_alarm < .5].index, cols].mean(),
        "No Fit, false alarm (> .5)": X_a2.loc[false_alarm[false_alarm > .5].index, cols].mean()}
out = pd.DataFrame(rows).T.round(2)
out.insert(0, "n", [int((miss < .5).sum()), int((miss > .5).sum()), int((false_alarm < .5).sum()), int((false_alarm > .5).sum())])
out

,n,role_match,experience_compatible,tfidf_scaled,coverage,coverage_unweighted,log_jd_concepts,log_cv_concepts,exp_known
"Fit, ranked correctly (miss < .5)",317,0.52,0.97,0.60,0.20,0.21,1.87,1.73,0.29
"Fit, missed (miss > .5)",16,0.03,0.81,0.19,0.02,0.02,1.43,1.65,0.38
"No Fit, ranked correctly",191,0.07,0.94,0.22,0.05,0.06,1.90,1.38,0.28
"No Fit, false alarm (> .5)",23,0.29,1.00,0.74,0.09,0.09,1.47,1.40,0.22


## Check 3 · Why were they misordered? (the LLM's own rationale for each pair)

The final set keeps one LLM series, and each of its 600 pairs has a short note saying why *it* chose its label (diagnostic reading only, D43; it is not a human reason). We group the notes of the pairs that were misordered by the hand-set X2 and count which kind of reason they give. The reason categories below are **exploratory** and were written before reading the notes; they come from regexes, so they are a rough count. No resume text is printed, and every note goes through \`annotation.redact\`.

In [7]:
import re
from candidate_screener.annotation import redact
notes = aw.llm_notes(frame)
notes["note"] = redact.redact(notes.notes)
CUES = {"different profession / role": r"different (profession|role|field|domain)|not a |instead of|rather than|mismatch|wrong role",
        "missing core skill": r"\b(lacks?|missing|no evidence|not mentioned|no experience|absent|without)\b",
        "seniority / years": r"\b(senior|junior|years?|yrs?|level|experience)\b|overqualified|under-?qualified",
        "partial / needs screening": r"partial|some |but |needs? (a )?(screening|clarification)|unclear|limited|potential",
        "strong match": r"strong|excellent|solid|matches|aligned|good fit"}
miss = base.groupby("fit_id").inv.mean(); fa = base.groupby("nofit_id").inv.mean()
groups = {"Fit, ranked correctly": miss[miss < .5].index, "Fit, missed": miss[miss > .5].index,
          "No Fit, ranked correctly": fa[fa < .5].index, "No Fit, false alarm": fa[fa > .5].index}
rows = {g: {c: notes.loc[ix, "note"].map(lambda x, p=p: bool(re.search(p, x, re.I))).mean() for c, p in CUES.items()} for g, ix in groups.items()}
t = pd.DataFrame(rows).T; t.insert(0, "n", [len(ix) for ix in groups.values()]); t.round(2)

,n,different profession / role,missing core skill,seniority / years,partial / needs screening,strong match
"Fit, ranked correctly",317,0.03,0.59,0.55,0.48,0.17
"Fit, missed",16,0.06,0.38,0.81,0.38,0.31
"No Fit, ranked correctly",191,0.45,0.24,0.60,0.09,0.05
"No Fit, false alarm",23,0.26,0.39,0.70,0.26,0.04


In [8]:
# does the LLM's own label agree with the final label on the misordered documents?
agree = {g: (notes.loc[ix, "llm_label"].isin(["Good Fit", "Potential Fit"]).astype(int) == lab.loc[ix, "y"]).mean() for g, ix in groups.items()}
pd.Series(agree, name="LLM label = final label").round(2).to_frame()

,LLM label = final label
"Fit, ranked correctly",0.87
"Fit, missed",0.75
"No Fit, ranked correctly",0.82
"No Fit, false alarm",0.57


### Fit CVs the score missed (up to 5, shortest 220 characters of each note)

In [9]:
def show(ix, k=5):
    for pid in list(ix)[:k]:
        r, x = notes.loc[pid], X_a2.loc[pid]
        print(f"[{lab.loc[pid, 'stratum']}] role {x.role_match:.2f} tfidf {x.tfidf_scaled:.2f} cov {x.coverage:.2f} | LLM {r.llm_label} {r.llm_score}")
        print("   " + r.note[:220])
show(groups["Fit, missed"])

[targeted] role 0.02 tfidf 0.25 cov 0.00 | LLM Potential Fit 52.0
   Analyst with relevant design/consulting skills, but lacks customer/technical support and JavaScript experience.
[targeted] role 0.01 tfidf 0.20 cov 0.00 | LLM Potential Fit 50.0
   Oracle DBA (20yrs) with strong SQL/ETL skills; missing 4+ years DBT, BigQuery, Snowplow, Kafka, and modern data stack.
[targeted] role 0.02 tfidf 0.00 cov 0.00 | LLM Potential Fit 38.0
   Python/AWS skills relevant but no data engineering or specific tools (DBT, BigQuery, Snowplow, Neo4J, Kafka) evidenced.
[generic] role 0.01 tfidf 0.00 cov 0.00 | LLM No Fit 28.0
   No professional development experience; only training courses completed
[generic] role 0.02 tfidf 0.00 cov 0.00 | LLM No Fit 20.0
   CV shows only web backend (Gin, MongoDB, Redis); no blockchain, Cosmos-SDK, systems programming, or crypto experience.


### No Fit CVs the score ranked high (up to 5)

In [10]:
show(groups["No Fit, false alarm"])

[generic] role 0.82 tfidf 0.00 cov 0.00 | LLM Potential Fit 40.0
   React developer without Angular experience or team leadership background. Core activity (team lead) not evidenced.
[targeted] role 0.03 tfidf 0.82 cov 0.00 | LLM Potential Fit 48.0
   Core engineering activity evident but thinly; missing required tools (DBT, BigQuery, etc.), years, education
[generic] role 0.19 tfidf 0.85 cov 0.49 | LLM Potential Fit 41.0
   Shows core activity thinly (pet project); career stage below 2yr requirement
[targeted] role 0.91 tfidf 0.64 cov 0.00 | LLM No Fit 28.0
   Entry-level candidate explicitly seeking junior role; position requires 2+ years BA experience
[generic] role 0.13 tfidf 0.98 cov 0.00 | LLM No Fit 12.0
   Graphic designer with no software engineering experience. Different profession from role requirements.


## Lead 1 · Is the concept vocabulary the gap? (label-free)

Check 3 found that most missed Fits and false alarms have coverage 0.00, and that the LLM's notes name tools such as DBT, BigQuery and Kafka. **Test (no labels read, exploratory):** how often does the coverage term have anything to work with, and does the extractor's vocabulary know the tools the notes name?

In [11]:
from candidate_screener.matching.entity_weighted import ConceptExtractor, normalized_phrase
pairs = corpus.a2_pairs(text_variant="v2")
ex = ConceptExtractor(); ex.concepts("python", "jd")             # builds the vocabulary
vocab = set(ex.aliases)
jd_c = pairs.jd_text.map(lambda t: ex.concepts(t, "jd")); cv_c = pairs.cv_text.map(lambda t: ex.concepts(t, "cv"))
n_jd, n_cv = jd_c.map(len), cv_c.map(len)
ov = np.array([len(a & b) for a, b in zip(jd_c, cv_c)])
jd_u = pairs.drop_duplicates("jd_id").index
pd.Series({"vocabulary aliases": len(vocab), "canonical concepts": len(set(ex.aliases.values())),
           "JDs": len(jd_u), "median concepts per JD": n_jd[jd_u].median(), "JDs with <= 3 concepts": (n_jd[jd_u] <= 3).mean(),
           "JDs with 0 concepts": (n_jd[jd_u] == 0).mean(), "median concepts per CV": n_cv.median(), "CVs with 0 concepts": (n_cv == 0).mean(),
           "pairs with no JD-CV concept overlap": (ov == 0).mean()}, name="A2 final, 600 pairs").round(2).to_frame()

,"A2 final, 600 pairs"
vocabulary aliases,9126.00
canonical concepts,1691.00
JDs,60.00
median concepts per JD,6.00
JDs with <= 3 concepts,0.33
JDs with 0 concepts,0.03
median concepts per CV,5.00
CVs with 0 concepts,0.13
pairs with no JD-CV concept overlap,0.53


In [12]:
# tool-like terms in the LLM notes (CI/CD and .NET are matched whole), and whether the vocabulary knows them
TECH = re.compile(r"CI/CD|\.NET|\b(?:[A-Z][a-z]+[A-Z][A-Za-z0-9]*|[A-Z]{2,}[A-Za-z0-9+#]*|[A-Za-z]+\+\+|C#|[A-Za-z]+\.js|[A-Za-z]+\d+)\b")
NOT_TOOLS = {"CV", "JD", "QA", "IT", "HR", "UK", "US", "EU", "CEO", "CTO", "PM", "BA", "MS", "AI", "ML", "SQL", "MUST", "HAVE", "LinkedIn"}
jd_text = pairs.set_index("pair_id").jd_text
rows = []
for pid, r in notes.iterrows():
    for term in set(TECH.findall(r.notes)) - NOT_TOOLS:
        rows.append({"term": term, "known": normalized_phrase(term) in vocab,
                     "in_jd": normalized_phrase(term) in normalized_phrase(jd_text.loc[pid])})
u = pd.DataFrame(rows).groupby("term").agg(known=("known", "first"), in_jd=("in_jd", "max"), n=("term", "size"))
j = u[u.in_jd]
print(f"{len(u)} distinct tool-like terms in the notes; {len(j)} also appear in the JD text; {j.known.mean():.0%} of those are in the vocabulary")
print("named most, vocabulary misses:", ", ".join(f"{k} x{v}" for k, v in j[~j.known].n.sort_values(ascending=False).head(20).items()))
print("named most, vocabulary knows: ", ", ".join(j[j.known].n.sort_values(ascending=False).head(10).index))

106 distinct tool-like terms in the notes; 78 also appear in the JD text; 38% of those are in the vocabulary
named most, vocabulary misses: CI/CD x23, UI x16, JS x15, CRM x13, BD x10, API x9, APIs x9, SDK x8, MEAN x8, UE5 x8, DBT x7, DWH x6, MSSQL x5, LLM x5, DW x5, ASP x4, SRS x4, SRE x4, MVC x3, ExpressJS x3
named most, vocabulary knows:  AWS, ETL, BI, DevOps, .NET, JavaScript, PHP, Node.js, UX, TypeScript


**Reading.** (1) For about half of the pairs the coverage term has nothing to work with (no JD-CV overlap), and a third of JDs yield three concepts or fewer. (2) The common stack is recognised, but short forms and the long tail are not (e.g. `ci/cd`, `api`, `dbt`, `sdk`, `crm`, `llm`, `mssql`, `ue5`). The match is whole-term against the alias list, so a mention the context filter drops is not counted here. (3) Next test, on D50 and not on the final set: add the aliases, or swap in GLiNER, re-measure the no-overlap rate, and rerun the learning curve.